# Project Horizon
- Klare Funktionen, die Datensammeln, aufbereiten (keine gaps?) und wegspeichern (erstmal csv)
    - Zeitraum soll klar angegeben werden können

In [55]:
import requests
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import pandas as pd

In [56]:
def get_timestamps(filter, resolution, region = "DE", ):
    url = f"https://www.smard.de/app/chart_data/{filter}/{region}/index_{resolution}.json"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

In [ ]:
#wasser_data = get_timestamps(1226, "quarterhour", "DE")

In [ ]:
## Easy unix milli timestamp to datetime conversion
#pd.to_datetime(wasser_data["timestamps"][0], unit="ms", utc=True).tz_convert("Europe/Berlin")

Timestamp('2014-12-29 00:00:00+0100', tz='Europe/Berlin')

In [59]:
relevant_filter = [1225, 1226, 1228, 4066, 4067, 4068, 4070, 410, 4359, 4387, 4169, 3791, 123, 125, 715, 5097, 122]
filter_meaning = ["wind_off", "water", "else_renew", "biomass", "wind_on", "solar", "pump", "netzlast", "residual", "pump_verbr", "preis", "prog_off", "prog_on", "prog_solar", "prog_else", "prog_wind_solar", "prog_all" ]

df_filter = pd.DataFrame({"filter": relevant_filter, "meaning": filter_meaning})

In [ ]:
#Für später wenn ich dataframe "fertig" habe
# def convert_timestamp_to_datetime():
#     utc_ms = preis_data['timestamps'][0]

    
#     utc_dt = datetime.fromtimestamp(utc_ms / 1000, tz=timezone.utc)

    
#     local_dt = utc_dt.astimezone(ZoneInfo("Europe/Berlin"))
#     print(type(local_dt))
#     print(local_dt)


In [ ]:
def get_values_from_tstamp(filter, resolution, timestamps_list, region = "DE"):
    comp_list = []
    for timestamp in timestamps_list:
        response = requests.get(f"https://www.smard.de/app/chart_data/{filter}/{region}/{filter}_{region}_{resolution}_{timestamp}.json")
        if response.status_code == 200:
            data = response.json()
            comp_list.extend(data["series"])

    return comp_list

In [74]:
def create_data_table():
    dt_json = {}
    for i in range(len(df_filter)):
        filter = df_filter.iloc[i,0]
        timestamps = get_timestamps(filter, "quarterhour", "DE")
        meaning = df_filter.iloc[i,1]
        dt_json[meaning] = get_values_from_tstamp(filter, "quarterhour", timestamps["timestamps"][-3:], "DE" )


        return dt_json
        

In [ ]:
data = create_data_table()

In [65]:
# def check_and_clean_data(data_dict):
#     for key, pairs in data_dict.items():
#         data_dict[key] = sorted(
#             (pair for pair in pairs if pair[1] is not None),
#             key=lambda pair: pair[0],
#         )
#     return data_dict
